# Challenge Three - Testing and Evaluation
**Goal:** Demonstrate the ability to write tests for and evaluate responses from large language models.


Setup and import dependences:


In [2]:
pip install google-genai pytest ipytest pandas


Setup our config:

In [8]:
import os
import pandas as pd

from google import genai
from google.genai import types


PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-04-c5701ae3d55f")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
MODEL = "gemini-2.5-flash"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

print(f"Ready - project={PROJECT_ID}, location={LOCATION}, model={MODEL}")

Ready - project=qwiklabs-gcp-04-c5701ae3d55f, location=us-central1, model=gemini-2.5-flash


Function one - question classification (deterministic)

In [11]:
VALID_CATEGORIES = {"Employment", "General Information", "Emergency Services", "Tax Related"}

CLASSIFY_SYSTEM_INSTRUCTION = """You are a routing classifier for the town of Aurora Bay, Alaska.
Classify each citizen question into EXACTLY ONE of these categories:
- Employment
- General Information
- Emergency Services
- Tax Related

Rules:
- Output ONLY the category name, exactly as written above. No punctuation, no explanation.
- "Emergency Services" covers police, fire, ambulance, search-and-rescue, and active hazards.
- "Tax Related" covers property tax, sales tax, payments, assessments, and exemptions.
- "Employment" covers town jobs, hiring, applications, and benefits for town workers.
- Anything else (hours, locations, events, services, general how-to) is "General Information".
"""


def classify_question(question: str) -> str:
    """Classify a citizen question into one of the four Aurora Bay service categories.

    Returns one of: 'Employment', 'General Information', 'Emergency Services', 'Tax Related'.
    """
    response = client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=CLASSIFY_SYSTEM_INSTRUCTION,
            temperature=0.0,
            max_output_tokens=20,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    result = (response.text or "").strip()

    # Normalize to the canonical category string so tests are robust to minor formatting drift.
    for category in VALID_CATEGORIES:
        if result.lower() == category.lower():
            return category
    return result  # return raw text on a miss so a failing test shows what the model said



Employment           <- How do I apply for a snow plow driver position with the town?
Emergency Services   <- There's a gas leak on Birch Street, who do I call?
Tax Related          <- When is the property tax payment due this year?
General Information  <- What time does the Aurora Bay community center open?


Function two - social media post generation (indeterminate):

In [14]:
POST_SYSTEM_INSTRUCTION = """You write official social media posts for the town of Aurora Bay, Alaska.
Rules for every post:
1. Keep the post under 280 characters.
2. Use a clear, calm, public-service tone.
3. End the post with the hashtag #AuroraBay
Output only the post text."""


def generate_post(announcement: str) -> str:
    """Generate a social media post for a government announcement.

    `announcement` is a short description of what to announce, e.g.
    'Heavy snow expected Thursday, all town schools closed.'
    """
    response = client.models.generate_content(
        model=MODEL,
        contents=announcement,
        config=types.GenerateContentConfig(
            system_instruction=POST_SYSTEM_INSTRUCTION,
            temperature=0.7,
            max_output_tokens=200,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return (response.text or "").strip()




LLM-as-judge helper:

In [15]:
def post_follows_rules(post: str) -> str:
    """Ask Gemini whether a post obeys the Aurora Bay posting rules. Returns 'Yes' or 'No'."""
    judge_prompt = f"""Does the social media post follow ALL of these rules?
1. The post is under 280 characters.
2. The post ends with the hashtag #AuroraBay

Answer with ONLY the single word Yes or No.

Post: {post}
Answer:"""
    response = client.models.generate_content(
        model=MODEL,
        contents=judge_prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=20,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return (response.text or "").strip().rstrip(".").capitalize()



Unit testing with pytest:

In [19]:
%%ipytest -qq

import pytest


class TestClassifyQuestion:
    @pytest.mark.parametrize("question,expected", [
        ("How do I apply for a job with the public works department?", "Employment"),
        ("My street is flooding and the power is out, who do I call?", "Emergency Services"),
        ("What is the deadline to pay my property taxes?", "Tax Related"),
        ("What are the hours for the public library?", "General Information"),
    ])
    def test_routes_to_expected_category(self, question, expected):
        assert classify_question(question) == expected

    def test_output_is_always_a_valid_category(self):
        result = classify_question("Is the ice rink open on weekends?")
        assert result in VALID_CATEGORIES


class TestGeneratePost:
    def test_post_under_280_chars_python_check(self):
        post = generate_post("Boil-water advisory in effect for downtown until further notice.")
        assert len(post) < 280

    def test_post_ends_with_hashtag_python_check(self):
        post = generate_post("Free sandbags available at the fire station this weekend.")
        assert post.endswith("#AuroraBay")

    def test_generated_post_passes_llm_judge(self):
        post = generate_post("Town council meeting moved to next Tuesday at 7 PM.")
        assert post_follows_rules(post) == "Yes"

    def test_judge_rejects_a_rule_breaking_post(self):
        bad_post = "Town meeting Tuesday."  # no hashtag -> should fail
        assert post_follows_rules(bad_post) == "No"

.........                                                                                    [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1345
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1345: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html


Evaluation with the Gen AI Evaluation Service:



In [26]:
# Computed metrics (ROUGE, BLEU, exact match) are pure n-gram math — no autorater,
# so they sidestep the JSON-parsing bug in the model-based path. They require a
# `reference` column (ground-truth answer) and a `response` column (what we evaluate).
#
# We use "bring your own response" (BYOR): instead of letting the eval service
# generate from bare prompts (which produced chat-reply menus, not posts), we call
# our own generate_post() so we're evaluating the function we actually built and tested.

# Reference posts: concise, compliant "ideal" versions for each announcement.
references = [
    "Heavy snow expected Thursday. All Aurora Bay schools are closed. Stay safe and avoid non-essential travel. #AuroraBay",
    "Town offices will be closed Monday for the holiday. Normal hours resume Tuesday. #AuroraBay",
    "Water main break on Spruce Ave. Crews are on site; expect low water pressure in the area. #AuroraBay",
    "Join us for the annual Winter Festival this Saturday at the town square, 10 AM to 4 PM. #AuroraBay",
    "The burn ban has been lifted effective today across the Aurora Bay district. #AuroraBay",
]


responses = [generate_post(a) for a in announcements]

# BYOR dataset: prompt + the response we generated + the reference to score against.
eval_df = pd.DataFrame({
    "prompt": announcements,
    "response": responses,
    "reference": references,
})

eval_df

,prompt,response,reference
0,Heavy snowfall expected Thursday. All Aurora B...,"Due to heavy snowfall, all Aurora Bay schools ...",Heavy snow expected Thursday. All Aurora Bay s...
1,Town offices closed Monday for the holiday.,Aurora Bay town offices will be closed on Mond...,Town offices will be closed Monday for the hol...
2,"Water main break on Spruce Ave; crews on site,...",Crews are responding to a water main break on ...,Water main break on Spruce Ave. Crews are on s...
3,Annual winter festival this Saturday at the to...,Join us for the annual Winter Festival this Sa...,Join us for the annual Winter Festival this Sa...
4,Burn ban lifted effective today across the Aur...,"Effective today, the burn ban for the Aurora B...",The burn ban has been lifted effective today a...


Evaluate generated responses against the references using computed metrics.

In [27]:
# Evaluate our generated responses against the references using computed metrics.

result = eval_client.evals.evaluate(
    dataset=eval_df,
    metrics=[
        vtypes.Metric(name="rouge_1"),
        vtypes.Metric(name="rouge_l_sum"),
        vtypes.Metric(name="bleu"),
        vtypes.Metric(name="exact_match"),
    ],
)
result.show()

Computing Metrics for Evaluation Dataset: 100%|██████████| 20/20 [00:01<00:00, 10.08it/s]


In [22]:
# Generate responses for two prompt variants by prepending different instruction framings.
# Variant A: terse, rules-first. Variant B: tone-first, more descriptive.

VARIANT_A = ("Write a social media post under 280 characters for this town announcement. "
             "End with #AuroraBay. Announcement: ")
VARIANT_B = ("You are the friendly voice of Aurora Bay. Turn this announcement into a calm, "
             "reassuring post (under 280 chars) that ends with #AuroraBay. Announcement: ")

df_a = pd.DataFrame({"prompt": [VARIANT_A + a for a in announcements]})
df_b = pd.DataFrame({"prompt": [VARIANT_B + a for a in announcements]})

inference_a = eval_client.evals.run_inference(model=MODEL, src=df_a)
inference_b = eval_client.evals.run_inference(model=MODEL, src=df_b)

inference_a.show()

Gemini Inference: 100%|██████████| 5/5 [00:10<00:00,  2.11s/it]
